In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import torch.nn as nn
import numpy as np
import os
import matplotlib
from matplotlib import animation 
from IPython.display import HTML 
import matplotlib.pyplot as plt
import copy
os.environ["MUJOCO_GL"] = "egl"
from dm_control import suite
import glob
from torch.utils.data import DataLoader, Dataset
from models import VAE, vae_loss
import math
device

device(type='cuda')

MDN-RNN
grabs z_t (encoder), gets a_t, predicts (zt+1) and r_t

In [10]:
ep_99 = np.load('/home/irusu/world_model/train_encodings_cartpole_swingup/episode_99.npz')

In [14]:
ep_99t = torch.from_numpy(ep_99['rewards']).float()

In [15]:
ep_99t.shape

torch.Size([125])

In [9]:
ep99['actions'].shape

(125, 1)

In [ ]:
train_encodings_cartpole_swingup

In [16]:
class LatentDataset(Dataset):
    "whole episode can be fed in as a single input, since there are only 125 steps in one"
    def __init__(self, folder):
        self.folder = folder
        
    def __len__(self):
        return len(os.listdir(self.folder)) # how many files in the folder
    
    
    def __getitem__(self, idx):
        """inputs are episodes[:-1], have to be concatenated to actions, 
        files are already permuted
        idx is the file!"""
        file_list_sorted = sorted(os.listdir(self.folder))
        file = file_list_sorted[idx]
        with np.load(os.path.join(self.folder, file)) as f:
            episode_z_mu = torch.from_numpy(f['z_mu']).float() #whole tensor shape 126, 32
            episode_z_logvar = torch.from_numpy(f['z_logvar']).float()   #whole tensor         
            episode_actions = torch.from_numpy(f['actions']).float()
            episode_rewards = torch.from_numpy(f['rewards']).float()
    # on each tensor of z_mu and z_logvar perform reparameterization to get z 
        sigma = torch.sqrt(torch.exp(episode_z_logvar)) #standard deviation
        eps = torch.randn_like(sigma) #random noise, from distribution of same shape as sigma
        z = episode_z_mu + sigma * eps
    # concatenate the z to action
        episode_tensor = torch.cat((z[:-1], episode_actions), dim = 1)
        target_tensor = z[1:]


        return episode_tensor, target_tensor, episode_rewards


In [28]:
TrainEpisodeDataset = LatentDataset('/home/irusu/world_model/train_encodings_cartpole_swingup')
TestEpisodeDataset = LatentDataset('/home/irusu/world_model/test_encodings_cartpole_swingup')

In [ ]:
#check it right outputs!
episode, target, rew = next(iter(EpisodeDataset))

In [25]:
ep_0 = EpisodeDataset[0][0]

In [26]:
ep_1 = EpisodeDataset[0][0]

In [27]:
ep_0 == ep_1

tensor([[False, False, False,  ..., False, False,  True],
        [False, False, False,  ..., False, False,  True],
        [False, False, False,  ..., False, False,  True],
        ...,
        [False, False, False,  ..., False, False,  True],
        [False, False, False,  ..., False, False,  True],
        [False, False, False,  ..., False, False,  True]])

In [22]:
episode.shape

torch.Size([125, 33])

In [ ]:
TrainDataLoader = DataLoader(TrainEpisodeDataset, batch_size = 16, shuffle = True)
TestDataLoader = DataLoader(TestEpisodeDataset, batch_size = 20, shuffle = False) #20 examples for test

In [43]:
class MDNRNN(nn.Module):
    def __init__(self, z_dim = 32, action_dim = 1, hidden_size = 256, K = 5): #z + 1 (action) -> 33 dims now
        super().__init__()
        self.z_dim = z_dim
        self.K = K
        self.action_dim = action_dim
        self.hidden_size = hidden_size
        input_dim = self.z_dim + self.action_dim
        guassian_outs = 2 * (self.K * self.z_dim) + self.K 
        self.lstm_l1 = nn.LSTM(input_size = input_dim, hidden_size = self.hidden_size, batch_first = True) #batch first changes layut of input and output tensors
        self.fc_mdn = nn.Linear(self.hidden_size, guassian_outs) # takes in h, gives back mixture params
        self.fc_rew = nn.Linear(self.hidden_size, 1)

    # MDN head is just a linear layer

    def forward(self, x):
        h, _ = self.lstm_l1(x)
        mdn_votes = self.fc_mdn(h) # B, T(time step), 325 gussian 
        rew = self.fc_rew(h)
        # slicing 
        pi_k, mu_k, sigma_k  = torch.split(mdn_votes, [self.K, self.K * self.z_dim, self.K * self.z_dim], dim=2)# first 5 are gaussian
        #reshape mu_k and sigma_k currently B,T, 160 into B, T, K, z_dim 
        B, T, _ = mu_k.shape
        mu_k = mu_k.view(B, T, self.K, self.z_dim)
        log_sigma_k = sigma_k.view(B, T, self.K, self.z_dim)
        # sigma_k = torch.exp(log_sigma_k) to get sigma 
        #softmax over pi_k to convert votes to probability
        log_pi_k = torch.log_softmax(pi_k, dim = -1) #B, T, K (5)
        return log_pi_k, mu_k, log_sigma_k, rew

    


In [48]:
# single batch test time
mdnrnn = MDNRNN()
# put the model on the device
mdnrnn.to(device)
optimizer = torch.optim.Adam(mdnrnn.parameters(), lr = 0.001)
episode_tensor, target_tensor, episode_rewards = next(iter(TrainDataLoader))
batch1 = episode_tensor.to(device)
target_tensor = target_tensor.to(device)
target_tensor = target_tensor.unsqueeze(2) # makes it B, T, 1, 32
episode_rewards = episode_rewards.to(device)
#run one batch 
loss_history = []
for step in range(100):
    log_pi_k, mu_k, log_sigma_k, rew = mdnrnn(batch1)
    #given the distribution M predicted, how probable is the real next z
    # target is B, T, 32, mu_k/ log_sigma_k are both B, T, 5, 32
    # for each B,T, we want 1 number, how probably is the real next z under the mixture M predicted
    # each gaussian predicts a mean and variance for each dimension in z
    
    # compute log N = −½·log(2π) − log σ − (z − μ)² / (2σ²) for each gaussian for each dimension
    nlog = -0.5 * math.log(2*math.pi) - log_sigma_k- ((target_tensor- mu_k)**2)/ (2*torch.exp(log_sigma_k)**2)
    
    #combine each dimension is 32, get 1 number for each of the 5 gaussians B, T, K
    log_prob_k = torch.sum(nlog, dim = -1)
    #mix the gaussians using their weights B, T. have to sum probabilities
    log_p_z= torch.logsumexp((log_prob_k + log_pi_k), dim = -1)
    mdn_loss = -torch.mean(log_p_z)
    reward_loss = torch.nn.functional.mse_loss(rew.squeeze(-1) , episode_rewards)
    total_loss = mdn_loss + reward_loss
    optimizer.zero_grad() # clear gradients from the previous step
    total_loss.backward()
    optimizer.step()
    batch_dic = {
        'total_loss': total_loss.item(), #stored as items 
        'mdn_loss': mdn_loss.item(),
        'reward_loss': reward_loss.item()
    }
    loss_history.append(batch_dic) #indent inside
    if step % 20 == 0: 
                print(f"step {step:3d} | total {total_loss.item():.3f} | mdn {mdn_loss.item():.3f} | reward {reward_loss.item():.3f}")



step   0 | total 47.762 | mdn 45.022 | reward 2.739
step  20 | total 45.138 | mdn 43.674 | reward 1.463
step  40 | total 43.222 | mdn 42.024 | reward 1.198
step  60 | total 41.426 | mdn 40.380 | reward 1.045
step  80 | total 39.899 | mdn 39.052 | reward 0.847


In [ ]:
def mdnrnn_loss():

torch.Size([64, 125, 32])